# Task 02 Solution: RAG Pipeline

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


In [ ]:
import json, os

# In Jupyter, os.getcwd() returns the notebook's directory (learning/, tasks/, solutions/)
# Fixtures are one level up at ../fixtures/input/
fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))

with open(os.path.join(fixtures, "tickets.json")) as f:
    tickets = json.load(f)

with open(os.path.join(fixtures, "knowledge_base.json")) as f:
    knowledge_base = json.load(f)

with open(os.path.join(fixtures, "test_questions.json")) as f:
    test_questions = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets, {len(knowledge_base)} KB articles, {len(test_questions)} test questions")


## Part 1: Vector Store

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

texts = [article["content"] for article in knowledge_base]
metadatas = [{"id": a["id"], "title": a["title"]} for a in knowledge_base]

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
vectorstore = FAISS.from_texts(texts, embeddings_model, metadatas=metadatas)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"✓ Indexed {len(texts)} articles")


In [ ]:
assert hasattr(retriever, 'invoke')
print("✓ retriever defined")

refund_docs = retriever.invoke("How long does a refund take?")
titles = [doc.metadata.get("title", "") for doc in refund_docs]
print(f"Retrieved: {titles}")
assert any("refund" in t.lower() or "policy" in t.lower() for t in titles)
print("✓ Retriever returns relevant documents")


## Part 2: RAG Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(
        f"[{i+1}] {doc.metadata['title']}\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )

rag_prompt = ChatPromptTemplate.from_template("""\
Answer the customer's question using ONLY the provided knowledge base articles.
If the answer is not in the articles, say "I don't have that information."
Be specific and cite relevant details.

Knowledge Base Articles:
{context}

Customer Question: {question}

Answer:""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("How long does a refund take to process?")
print(answer)


In [ ]:
assert isinstance(answer, str) and len(answer) > 20
assert any(kw in answer.lower() for kw in ["5-7", "business days", "refund"])
print("✓ RAG chain test passed")


## Part 3: Evaluation

In [ ]:
def evaluate_rag(rag_chain, test_questions):
    scores = []
    for q in test_questions:
        answer = rag_chain.invoke(q["question"]).lower()
        hits = sum(1 for kw in q["expected_keywords"] if kw.lower() in answer)
        scores.append(hits / len(q["expected_keywords"]))
    return sum(scores) / len(scores)

coverage = evaluate_rag(rag_chain, test_questions)
print(f"Coverage: {coverage:.0%}")


In [ ]:
for q in test_questions:
    answer = rag_chain.invoke(q["question"]).lower()
    hits = [kw for kw in q["expected_keywords"] if kw.lower() in answer]
    score = len(hits) / len(q["expected_keywords"])
    status = "✓" if score >= 0.5 else "✗"
    print(f"  {status} Q{q['id']}: {score:.0%} ({hits})")

assert coverage >= 0.60
print(f"\n✓ RAG evaluation passed ({coverage:.0%})")
